In [1]:
# ==========================
# IMPORT LIBRARIES
# ==========================

import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
# ==========================
# NLTK DOWNLOADS
# ==========================

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
# ==========================
# LOAD DATASET
# ==========================

df = pd.read_csv("HateSpeechData.csv")

print(df.head())
print(df.shape)

   Unnamed: 0  count  hate_speech  offensive_language  neither  class  \
0           0      3            0                   0        3      2   
1           1      3            0                   3        0      1   
2           2      3            0                   3        0      1   
3           3      3            0                   2        1      1   
4           4      6            0                   6        0      1   

                                               tweet  
0  !!! RT @mayasolovely: As a woman you shouldn't...  
1  !!!!! RT @mleew17: boy dats cold...tyga dwn ba...  
2  !!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...  
3  !!!!!!!!! RT @C_G_Anderson: @viva_based she lo...  
4  !!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...  
(24783, 7)


In [5]:
# ==========================
# PREPROCESSING
# ==========================

stop_words = set(stopwords.words("english"))

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)

    tokens = word_tokenize(text)

    tokens = [w for w in tokens if w not in stop_words]

    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    tokens = [stemmer.stem(w) for w in tokens]

    return " ".join(tokens)

print("Preprocessing Started...")

df["processed_text"] = df["tweet"].apply(preprocess)

print("Preprocessing Completed!")

Preprocessing Started...
Preprocessing Completed!


In [14]:

df["processed_text"] [:5]

0    rt woman shouldnt complain clean hous amp man ...
1    rt boy dat coldtyga dwn bad cuffin dat hoe st ...
2     rt dawg rt ever fuck bitch start cri confus shit
3                                  rt look like tranni
4    rt shit hear might true might faker bitch told ya
Name: processed_text, dtype: str

In [6]:
# ==========================
# FEATURES & LABELS
# ==========================

X = df["processed_text"]

y = df["class"]


In [7]:
# ==========================
# TF-IDF VECTORIZATION
# ==========================

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(X)

print("TF-IDF Shape:", X.shape)


TF-IDF Shape: (24783, 10000)


In [8]:
# ==========================
# TRAIN TEST SPLIT
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



In [9]:
# MODELS
# ==========================

models = {

    "Logistic Regression":
        LogisticRegression(max_iter=3000),

    "Naive Bayes":
        MultinomialNB(),

    "Linear SVM":
        LinearSVC(),

    "Decision Tree":
        DecisionTreeClassifier(random_state=42),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ),

    "Extra Trees":
        ExtraTreesClassifier(
            n_estimators=100,
            random_state=42
        ),

    "AdaBoost":
        AdaBoostClassifier(
            n_estimators=100,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            random_state=42
        ),

    "KNN":
        KNeighborsClassifier(
            n_neighbors=5
        )
}

In [10]:
# ==========================
# TRAIN & EVALUATE
# ==========================

results = []

for name, model in models.items():

    print(f"\nTraining {name} ...")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1
    ])


Training Logistic Regression ...

Training Naive Bayes ...

Training Linear SVM ...

Training Decision Tree ...

Training Random Forest ...

Training Extra Trees ...

Training AdaBoost ...

Training Gradient Boosting ...

Training KNN ...


In [11]:
# ==========================
# RESULTS TABLE
# ==========================

results_df = pd.DataFrame(
    results,
    columns=[
        "Algorithm",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n\n==========================")
print("MODEL COMPARISON")
print("==========================")

print(results_df)



MODEL COMPARISON
             Algorithm  Accuracy  Precision    Recall  F1 Score
4        Random Forest  0.900141   0.886658  0.900141  0.885233
0  Logistic Regression  0.896107   0.885602  0.896107  0.880502
5          Extra Trees  0.894089   0.879252  0.894089  0.879596
2           Linear SVM  0.891668   0.879822  0.891668  0.881612
7    Gradient Boosting  0.877749   0.865658  0.877749  0.864300
3        Decision Tree  0.872504   0.867452  0.872504  0.869780
1          Naive Bayes  0.840629   0.855458  0.840629  0.800384
6             AdaBoost  0.818035   0.789436  0.818035  0.781537
8                  KNN  0.623159   0.716382  0.623159  0.647142


In [12]:
# SAVE RESULTS
# ==========================

results_df.to_csv(
    "classification_results.csv",
    index=False
)

print("\nResults Saved Successfully!")


Results Saved Successfully!
